# 狼人 vs 村民 胜率统计

该 notebook 统计胜率并可视化：

- 优先使用 `logs/eval_results_*.csv`（若存在）
- 若无 CSV，则回退到 `logs/session_*/game_complete.json`

输出内容：
- 各阵营胜场数
- 胜率（%）
- 柱状图

In [ ]:
from __future__ import annotations

import json
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd


def project_root() -> Path:
    cwd = Path.cwd()
    if (cwd / "logs").is_dir():
        return cwd
    if (cwd.parent / "logs").is_dir():
        return cwd.parent
    return cwd


ROOT = project_root()
LOGS_DIR = ROOT / "logs"

In [ ]:
def latest_eval_csv(logs_dir: Path) -> Path | None:
    files = sorted(logs_dir.glob("eval_results_*.csv"))
    return files[-1] if files else None


def load_winners_from_eval_csv(csv_path: Path) -> list[str]:
    df = pd.read_csv(csv_path)
    # 兼容列名 Winner/winner
    if "Winner" in df.columns:
        winners = df["Winner"].dropna().astype(str).tolist()
    elif "winner" in df.columns:
        winners = df["winner"].dropna().astype(str).tolist()
    else:
        winners = []
    return winners


def iter_session_state_files(logs_dir: Path):
    for d in sorted(logs_dir.glob("session_*")):
        if not d.is_dir():
            continue
        complete = d / "game_complete.json"
        if complete.is_file():
            yield complete


def load_winners_from_sessions(logs_dir: Path) -> list[str]:
    winners: list[str] = []
    for path in iter_session_state_files(logs_dir):
        try:
            data = json.loads(path.read_text(encoding="utf-8"))
        except (OSError, json.JSONDecodeError):
            continue
        winner = data.get("winner")
        if winner:
            winners.append(str(winner))
    return winners


def normalize_winner(w: str) -> str | None:
    val = w.strip().lower()
    if val in {"villagers", "villager"}:
        return "Villagers"
    if val in {"werewolves", "werewolf"}:
        return "Werewolves"
    return None

In [ ]:
# 优先 CSV，其次 session 完整局
csv_path = latest_eval_csv(LOGS_DIR)
if csv_path is not None:
    raw_winners = load_winners_from_eval_csv(csv_path)
    source = f"eval csv: {csv_path.name}"
else:
    raw_winners = load_winners_from_sessions(LOGS_DIR)
    source = "session game_complete.json"

winners = [x for x in (normalize_winner(w) for w in raw_winners) if x is not None]
counts = Counter(winners)

villagers = counts.get("Villagers", 0)
werewolves = counts.get("Werewolves", 0)
total = villagers + werewolves

print(f"Data source: {source}")
print(f"Total completed games: {total}")
print(f"Villagers wins: {villagers}")
print(f"Werewolves wins: {werewolves}")

if total == 0:
    print("No completed games found.")
else:
    v_rate = 100.0 * villagers / total
    w_rate = 100.0 * werewolves / total
    print(f"Villagers win rate: {v_rate:.2f}%")
    print(f"Werewolves win rate: {w_rate:.2f}%")

    df = pd.DataFrame(
        {
            "Camp": ["Villagers", "Werewolves"],
            "Wins": [villagers, werewolves],
            "WinRate(%)": [v_rate, w_rate],
        }
    )
    display(df)

In [ ]:
if total > 0:
    labels = ["Villagers", "Werewolves"]
    wins = [villagers, werewolves]
    rates = [100.0 * villagers / total, 100.0 * werewolves / total]

    fig, ax = plt.subplots(figsize=(7, 4.5))
    bars = ax.bar(labels, rates, color=["#2a9d8f", "#e76f51"], edgecolor="black")
    ax.set_ylim(0, 100)
    ax.set_ylabel("Win Rate (%)")
    ax.set_title("Win Rate by Camp")

    for i, b in enumerate(bars):
        h = b.get_height()
        ax.annotate(
            f"{wins[i]} wins\n{h:.2f}%",
            xy=(b.get_x() + b.get_width() / 2, h),
            xytext=(0, 3),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontsize=9,
        )

    plt.tight_layout()
    plt.show()